In [12]:
import os
import pandas as pd
import requests ##Need pip install requests
import sys
import gzip
from pathlib import Path

In [ ]:
data_source = {
    "BioSNAP" : r"C:\GitHub Project\dda - backend\data\raw\BioSNAP",
    "Drug-Indication" : r"C:\GitHub Project\dda - backend\data\raw\Kaggle\Drug Indications (Drug Engineering with AI)",
    "Drug-SideEffect-MedicalConditions" : r"C:\GitHub Project\dda - backend\data\raw\Kaggle\Drugs, Side Effects, and Medical Conditions",
}

## Edit: choose BioSNAP, Loai bo Drug-Indication, Lua chon Drug-SideEffect-MedicalConditions


In [20]:
def analyst_data(data_source):
    for name, fie_path in data_source.items():
        print(f"---------------------Analysing {name} data...---------------------")
        for file in os.listdir(fie_path):
            print(f"-----------------Analysing {file}...-----------------")
            df = pd.read_csv(os.path.join(fie_path, file))
            print('Shape: ', df.shape)
            print('Columns: ', df.columns.tolist())
            print('Sample data:')
            print(df.head(10))
            print('-'*100)
            print()

In [21]:
analyst_data(data_source)

---------------------Analysing BioSNAP data...---------------------
-----------------Analysing DCh-Miner_miner-disease-chemical.tsv.gz...-----------------
Shape:  (466657, 1)
Columns:  ['# Disease(MESH)\tChemical']
Sample data:
  # Disease(MESH)\tChemical
0     MESH:D005923\tDB00564
1     MESH:D009503\tDB01072
2     MESH:D016115\tDB01759
3     MESH:D018476\tDB00451
4     MESH:C567059\tDB00641
5     MESH:D010198\tDB00481
6     MESH:D007898\tDB04173
7     MESH:C574275\tDB00636
8     MESH:D001249\tDB00814
9     MESH:C535836\tDB00091
----------------------------------------------------------------------------------------------------

---------------------Analysing Drug-Indication data...---------------------
-----------------Analysing drug_indications_database.csv...-----------------


C:\Users\ASPIRE A715 - 42G\AppData\Local\Temp\ipykernel_18356\2273263934.py:6: DtypeWarning: Columns (14,21,26,27,28,34,38,40,41,53,54,61,62,63) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(os.path.join(fie_path, file))


Shape:  (192833, 64)
Columns:  ['DID_id', 'src_nm', 'src_id', 'drug_raw_name', 'cas_cas_no', 'cas_pt', 'cas_source', 'cas_match', 'chebi_pt', 'chebi_id', 'chebi_cas_no', 'chebi_match', 'chebi_syn', 'chebi_syn_match', 'chebi_match_aid', 'chemid_pt', 'chemid_display_name', 'chemid_id', 'chemi_match', 'chemid_syn', 'chemid_syn_match', 'chemid_match_aid', 'ctd_pt', 'ctd_mesh_id', 'ctd_cas_no', 'ctd_match', 'ctd_syn', 'ctd_syn_match', 'ctd_match_aid', 'umls_pt', 'umls_cui', 'umls_match', 'umls_syn', 'umls_syn_match', 'umls_match_aid', 'umls_sem_typ1', 'umls_sem_typ2', 'umls_sem_typ3', 'umls_sem_typ4', 'ind_raw', 'ind_agg', '_ind_search', 'ind_raw_value', 'ind_raw_target', 'ind_raw_match', 'ind_umls_entry_term_match', 'ind_umls_entry_term', 'ind_umls_pt', 'und_umls_cuii', 'ind_umls_term_typ', 'ind_umls_pheno_flg', 'ind_umls_sem_typ1', 'ind_umls_sem_typ2', 'ind_umls_sem_typ3', 'ind_umls_sem_typ4', 'ind_umls_in_term_match', 'ind_umls_in_term', 'ind_umls_in_pt', 'ind_umls_in_cui', 'ind_umls_in_

In [36]:
## Simple analysis BioSNAP
file_path = r"C:\GitHub Project\dda - backend\data\raw\BioSNAP\DCh-Miner_miner-disease-chemical.tsv.gz"
df = pd.read_csv(
    file_path,
    sep='\t',
    compression='gzip'
)

print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
print()
print('Sample:')
print(df.head(10))
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Unique Drug IDs:', df['Chemical'].nunique())
print('Unique Disease IDs:', df['# Disease(MESH)'].nunique())


Shape: (466657, 2)
Columns: ['# Disease(MESH)', 'Chemical']

Sample:
  # Disease(MESH) Chemical
0    MESH:D005923  DB00564
1    MESH:D009503  DB01072
2    MESH:D016115  DB01759
3    MESH:D018476  DB00451
4    MESH:C567059  DB00641
5    MESH:D010198  DB00481
6    MESH:D007898  DB04173
7    MESH:C574275  DB00636
8    MESH:D001249  DB00814
9    MESH:C535836  DB00091

Missing values:
# Disease(MESH)    0
Chemical           0
dtype: int64

Unique Drug IDs: 1663
Unique Disease IDs: 5536


In [35]:
## Mapping by PubChem
# Test mapping DB ID -> tên thuốc qua PubChem
test_ids = ['DB00564', 'DB01072', 'DB01759', 'DB00451', 'DB00641']

for db_id in test_ids:
    url = f'https://pubchem.ncbi.nlm.nih.gov/rest/pug/substance/sourceid/drugbank/{db_id}/JSON'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        name = data['PC_Substances'][0]['synonyms'][0]
        print(f'{db_id} -> {name}')
    else:
        print(f'{db_id} -> Not found (status {response.status_code})')


# Test mapping MESH ID -> tên bệnh qua NIH API
test_ids = ['D005923', 'D009503', 'D016115', 'D018476', 'D001249']

for mesh_id in test_ids:
    url = f'https://id.nlm.nih.gov/mesh/{mesh_id}.json'
    response = requests.get(url)
    if response.status_code == 200:
        data = response.json()
        name = data['label']['@value']
        print(f'MESH:{mesh_id} -> {name}')
    else:
        print(f'MESH:{mesh_id} -> Not found (status {response.status_code})')

DB00564 -> Carbamazepine
DB01072 -> Atazanavir
DB01759 -> Kojic acid
DB00451 -> Levothyroxine
DB00641 -> Simvastatin
MESH:D005923 -> Glomerulosclerosis, Focal Segmental
MESH:D009503 -> Neutropenia
MESH:D016115 -> Albinism, Oculocutaneous
MESH:D018476 -> Hypokinesia
MESH:D001249 -> Asthma


In [1]:
##Test mapping
import sys
sys.path.append('../..')
from data.mapping_biosnap import fetch_drug_name, fetch_disease_name

# Test drug
print(fetch_drug_name('DB00564'))
print(fetch_drug_name('DB00641'))

# Test disease dang D
print(fetch_disease_name('MESH:D005923'))

# Test disease dang C
print(fetch_disease_name('MESH:C567059'))
print(fetch_disease_name('MESH:C574275'))

('DB00564', 'Carbamazepine')
('DB00641', 'Simvastatin')
('MESH:D005923', 'Glomerulosclerosis, Focal Segmental')
('MESH:C567059', 'Multiple Endocrine Neoplasia, Type IV')
('MESH:C574275', 'Ectrodactyly')


In [7]:
import gzip
from pathlib import Path
import pandas as pd

dataset_path = r"C:\GitHub Project\dda - backend\data\raw\BioSNAP"

ctd_diseases = pd.read_csv(
    Path(dataset_path, "CTD_diseases.csv.gz"),
    compression='gzip',
    skiprows=28,
    header=None,
    names=['DiseaseName','DiseaseID','AltDiseaseIDs','Definition',
           'ParentIDs','TreeNumbers','ParentTreeNumbers','Synonyms','SlimMappings'],
    usecols=['DiseaseName', 'DiseaseID']
)

print('Shape:', ctd_diseases.shape)
print('Columns:', ctd_diseases.columns.tolist())
print(ctd_diseases.head(5))

Shape: (13322, 2)
Columns: ['DiseaseName', 'DiseaseID']
                       DiseaseName     DiseaseID
0                                #           NaN
1  10p Deletion Syndrome (Partial)  MESH:C538288
2            13q deletion syndrome  MESH:C535484
3              15q24 Microdeletion  MESH:C579849
4        16p11.2 Deletion Syndrome  MESH:C579850


In [8]:
import pandas as pd
from pathlib import Path

dataset_path = r"C:\GitHub Project\dda - backend\data\raw\BioSNAP"

ctd_diseases = pd.read_csv(
    Path(dataset_path, "CTD_diseases.csv.gz"),
    compression='gzip',
    skiprows=29,
    header=None,
    names=['DiseaseName','DiseaseID','AltDiseaseIDs','Definition',
           'ParentIDs','TreeNumbers','ParentTreeNumbers','Synonyms','SlimMappings'],
    usecols=['DiseaseName', 'DiseaseID']
)

print('Shape:', ctd_diseases.shape)
print(ctd_diseases.head(5))

# Kiểm tra có match với BioSNAP không
biosnap = pd.read_csv(
    Path(dataset_path, "DCh-Miner_miner-disease-chemical.tsv.gz"),
    sep='\t',
    compression='gzip'
)
biosnap = biosnap.rename(columns={"# Disease(MESH)": "disease_id"})

ctd_lookup = dict(zip(ctd_diseases['DiseaseID'], ctd_diseases['DiseaseName']))
biosnap['disease_name'] = biosnap['disease_id'].map(ctd_lookup)

found = biosnap['disease_name'].notna().sum()
total = len(biosnap)
unique_found = biosnap[biosnap['disease_name'].notna()]['disease_id'].nunique()
unique_total = biosnap['disease_id'].nunique()

print(f"\nMatch rate: {found}/{total} pairs ({found/total*100:.1f}%)")
print(f"Unique diseases found: {unique_found}/{unique_total}")

Shape: (13321, 2)
                        DiseaseName     DiseaseID
0   10p Deletion Syndrome (Partial)  MESH:C538288
1             13q deletion syndrome  MESH:C535484
2               15q24 Microdeletion  MESH:C579849
3         16p11.2 Deletion Syndrome  MESH:C579850
4  17,20-Lyase Deficiency, Isolated  MESH:C567076

Match rate: 460127/466657 pairs (98.6%)
Unique diseases found: 5402/5536


In [ ]:


dataset_path = r"C:\GitHub Project\dda - backend\data\raw\BioSNAP"

with gzip.open(Path(dataset_path, "CTD_chemicals.csv.gz"), 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        print(f"Line {i}: {repr(line[:150])}")
        if i >= 32:
            break

Line 0: '# The Comparative Toxicogenomics Database (CTD) - http://ctdbase.org/\n'
Line 1: '#   Copyright 2002-2012 MDI Biological Laboratory. All rights reserved.\n'
Line 2: '#   Copyright 2012-2026 NC State University. All rights reserved.\n'
Line 3: '#  \n'
Line 4: '# \n'
Line 5: '# Use is subject to the terms set forth at http://ctdbase.org/about/legal.jsp\n'
Line 6: '# These terms include:\n'
Line 7: '# \n'
Line 8: '#   1. All forms of publication (e.g., web sites, research papers, databases,\n'
Line 9: '#      software applications, etc.) that use or rely on CTD data must cite CTD.\n'
Line 10: '#      Citation guidelines: http://ctdbase.org/about/publications/#citing\n'
Line 11: '# \n'
Line 12: '#   2. All electronic or online applications must include hyperlinks from \n'
Line 13: '#      contexts that use CTD data to the applicable CTD data pages.\n'
Line 14: '#      Linking instructions: http://ctdbase.org/help/linking.jsp\n'
Line 15: '# \n'
Line 16: '#   3. You must notify CTD,

In [13]:

df = pd.read_csv(r"C:\GitHub Project\dda - backend\data\processed\dda_dataset.csv")

# Xem negative pairs dùng những drug_id nào
negatives = df[df['label'] == 0]
print("Unique drug_id trong negative pairs:", negatives['drug_id'].nunique())
print("Sample drug_id:", negatives['drug_id'].head(10).tolist())

Unique drug_id trong negative pairs: 1654
Sample drug_id: ['DB00773', 'DB03435', 'DB00455', 'DB00350', 'DB01506', 'DB00670', 'DB00181', 'DB00492', 'DB00610', 'DB00576']


In [ ]:
print('Shape:', df.shape)
print()
print('Sample positive (label=1):')
print(df[df['label']==1][['drug_name','disease_name','label']].head(3).to_string())
print()
print('Sample negative (label=0):')
print(df[df['label']==0][['drug_name','disease_name','label']].head(3).to_string())
print()
print('Missing values:')
print(df.isnull().sum())

Shape: (933314, 5)

Sample positive (label=1):
      drug_name         disease_name  label
3     Berberine            Neuralgia      1
4     Docetaxel  Febrile Neutropenia      1
5  Alitretinoin   Carcinoma, Lobular      1

Sample negative (label=0):
                drug_name                          disease_name  label
0               Etoposide  RENAL-HEPATIC-PANCREATIC DYSPLASIA 1      0
1  Uridine-5'-Diphosphate                   Mouth Abnormalities      0
2              Loratadine        Orofaciodigital syndrome type1      0

Missing values:
drug_id            0
drug_name       1930
disease_id         0
disease_name    1218
label              0
dtype: int64


In [14]:
print('Trước khi drop:', df.shape)
df = df.dropna(subset=['drug_name', 'disease_name'])
print('Sau khi drop:', df.shape)

print()
print('Label distribution sau drop:')
print(df['label'].value_counts())

# Lưu lại
df.to_csv(r"C:\GitHub Project\dda - backend\data\processed\dda_dataset.csv", index=False)
print()
print('Đã lưu lại file.')

Trước khi drop: (933314, 5)
Sau khi drop: (930168, 5)

Label distribution sau drop:
label
0    466657
1    463511
Name: count, dtype: int64

Đã lưu lại file.
